## *RaschPy* simulation functionality

This notebook works through examples of how to generate simulated data sets with `RaschPy` for experimental use where knowledge of the underlying 'ground truth' of the generating parameters is useful, for example when comparing the efficacy of different estimation algorithms, such as in Elliott & Buttery (2022a) or exploring the effect of fitting different Rasch models to the same data set, such as in Elliott & Buttery (2022b). There are separate classes for each model: `SLM_Sim` for the simple logistic model (or dichotomous Rasch model) (Rasch, 1960), `PCM_Sim` for the partial credit model (Masters, 1982), `RSM_Sim` for the rating scale model (Andrich, 1978), `MFRM_Sim_Global` for the many-facet Rasch model (Linacre, 1994), and the family of extended MFRMs (Elliott 2025, Elliott & Buttery, 2022b): `MFRM_Sim_Items` for the vector-by-item extended MFRM, `MFRM_Sim_Thresholds` for the vector-by-threshold extended MFRM, `MFRM_Sim_Matrix` for the matrix extended MFRM, `MFRM_Sim_Bivector` for the bivector extended MFRM, and three closed-form stretch-restricted parameterisations: `MFRM_Sim_Centrality` (Jin & Wang 2018-style rater centrality/extremity, restricting `MFRM_Sim_Thresholds`), `MFRM_Sim_PseudoHalo` (an item-difficulty-compression model, restricting `MFRM_Sim_Items`), and `MFRM_Sim_Bistretch` (combining both stretch axes at once, restricting `MFRM_Sim_Bivector`). All data is generated to fit the chosen model.

**References**

&nbsp;&nbsp;&nbsp;&nbsp; Andrich, D. (1978). A rating formulation for ordered response categories. *Psychometrika*, *43*(4), 561–573.

&nbsp;&nbsp;&nbsp;&nbsp;   Elliott, M. (2025). Extended many-facet Rasch models: Accounting for rater effects in automated essay scoring systems [Apollo - University of Cambridge Repository]. https://doi.org/10.17863/CAM.127567

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M., & Buttery, P. J. (2022a) Non-iterative Conditional Pairwise Estimation for the Rating Scale Model, *Educational and Psychological Measurement*, *82*(5), 989-1019.

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M. and Buttery, P. J. (2022b) Extended Rater Representations in the Many-Facet Rasch Model, *Journal of Applied Measurement*, *22*(1), 133-160.

&nbsp;&nbsp;&nbsp;&nbsp; Jin, K.-Y., & Wang, W.-C. (2018). A new facets model for rater's centrality/extremity response style. *Journal of Educational Measurement*, *55*(4), 543–563.

&nbsp;&nbsp;&nbsp;&nbsp; Linacre, J. M. (1994). *Many-Facet Rasch Measurement*. MESA Press.

&nbsp;&nbsp;&nbsp;&nbsp; Masters, G. N. (1982). A Rasch model for partial credit scoring. *Psychometrika*, *47*(2), 149–174.

&nbsp;&nbsp;&nbsp;&nbsp; Rasch, G. (1960). *Probabilistic models for some intelligence and attainment tests*. Danmarks Pædagogiske
Institut.

Import the packages and set the working directory (here called `my_working_directory`) - you will save your output files here.

In [2]:
import raschpy as rp
import numpy as np
import pandas as pd
import os

os.chdir('my_working_directory')

### `MFRM_Sim_Centrality`

Create an object `mfrm_sim_1` of the class `MFRM_Sim_Centrality`. Unlike the fully-free `MFRM_Sim_Thresholds`, each rater's threshold-severity profile is generated from just two true parameters: a severity shift `lambda_r` and a threshold-stretch `omega_r` (Jin & Wang 2018-style rater centrality/extremity). Internally, `MFRM_Sim_Centrality` first builds a preliminary `MFRM_Sim_Thresholds` simulation to get realistic item locations, Rasch-Andrich thresholds and person locations -- so all the same `item_range`, `category_base`, `max_disorder`, `person_sd` and `offset` arguments apply here too, controlling that preliminary simulation. `global_range` controls the spread of the true `lambda_r` values (analogous to its role in the other MFRM simulations), and the new `stretch_range` argument controls the spread of the true `omega_r` values around 1 (`omega_r > 1` is 'central' -- reference thresholds are spread apart for that rater; `0 < omega_r < 1` is 'extreme' -- reference thresholds are compressed together). `lambda_r`/`omega_r` are sampled from a symmetric distribution by default; pass `manual_lambda`/`manual_omega` to specify them directly instead. We pass `item_range=4`, `global_range=3` (spread of true `lambda_r`), `stretch_range=1` (spread of true `omega_r` around 1), `category_base=1.5` and `max_disorder=1`, plus `person_sd=2` and `offset=0.5`. One other required argument is `max_score`. There are 500 persons, 8 items and 10 raters, with no missing data for this simulation.

In [3]:
mfrm_sim_1 = rp.MFRM_Sim_Centrality(no_of_items=8,
                                    no_of_persons=500,
                                    no_of_facet_elements=10,
                                    max_score=5,
                                    item_range=4,
                                    global_range=3,
                                    stretch_range=1,
                                    category_base=1.5,
                                    max_disorder=1,
                                    person_sd=2,
                                    offset=0.5,
                                    seed=42)

Save the generated response dataframe, which is stored as an attribute `mfrm_sim_1.responses`, to file, and view the first 5 lines.

In [4]:
mfrm_sim_1.responses.to_csv('mfrm_sim_1_responses.csv')
mfrm_sim_1.responses.head()

Item_1  Item_2  Item_3  Item_4  Item_5  Item_6  Item_7  \
Rater_1 Person_1       3       4       2       4       5       0       1   
        Person_2       3       2       1       1       1       0       0   
        Person_3       4       5       3       5       4       3       1   
        Person_4       4       5       4       5       5       2       2   
        Person_5       1       1       0       2       1       0       0   

                  Item_8  
Rater_1 Person_1       2  
        Person_2       1  
        Person_3       2  
        Person_4       2  
        Person_5       0

Save the generating item, threshold, rater and person parameters to file, and view the first 5 lines of the item locations, rater facet effects and person locations, plus the Rasch-Andrich thresholds.

In [5]:
mfrm_sim_1.items.to_csv('mfrm_sim_1_items.csv', header=None)
mfrm_sim_1.items.head()

Item_1   -0.775991
Item_2   -1.774624
Item_3    0.267996
Item_4   -1.929639
Item_5   -0.809525
dtype: float64

In [6]:
mfrm_sim_1.thresholds.to_csv('mfrm_sim_1_thresholds.csv', header=None)
mfrm_sim_1.thresholds

1   -3.021099
2   -1.710138
3   -0.022473
4    1.682934
5    3.070775
dtype: float64

In [7]:
mfrm_sim_1.facet_effects.to_csv('mfrm_sim_1_facet_effects.csv')
mfrm_sim_1.facet_effects.head()

,1,2,3,4,5
Rater_1,0.955563,0.737145,0.455965,0.171829,-0.059397
Rater_2,-1.769334,-1.192854,-0.450724,0.299208,0.909495
Rater_3,0.512998,0.618379,0.754042,0.891131,1.002693
Rater_4,-0.600519,-0.242878,0.217530,0.682778,1.061393
Rater_5,-1.243955,-1.375573,-1.545011,-1.716230,-1.855567


In [8]:
mfrm_sim_1.persons.to_csv('mfrm_sim_1_persons.csv', header=None)
mfrm_sim_1.persons.head()

Person_1    1.135686
Person_2   -1.553716
Person_3    2.027155
Person_4    2.407382
Person_5   -3.375818
dtype: float64

Unlike `MFRM_Sim_Thresholds`, `MFRM_Sim_Centrality` also exposes the true generating `lambda_r`/`omega_r` values directly (before they are expanded into the full `facet_effects` table above); view and save:

In [9]:
mfrm_sim_1.lambda_.to_csv('mfrm_sim_1_lambda.csv', header=None)
mfrm_sim_1.omega.to_csv('mfrm_sim_1_omega.csv', header=None)
round(pd.DataFrame({'lambda': mfrm_sim_1.lambda_, 'omega': mfrm_sim_1.omega}), 3)

,lambda,omega
Rater_1,0.452221,0.833391
Rater_2,-0.440842,1.439738
Rater_3,0.755849,1.080385
Rater_4,0.223661,1.272808
Rater_5,-1.547267,0.899602
Rater_6,1.452733,0.687643
Rater_7,0.411750,0.998624
Rater_8,0.491500,0.439738
Rater_9,-1.387699,1.279144
Rater_10,-0.411906,1.068927


View `max_score`.

In [10]:
mfrm_sim_1.max_score

5

Create an object `mfrm_1` of the class `MFRM` from the response dataframe for analysis. The new object `mfrm_1` automatically inherits all the parameters from `mfrm_sim_1`, storing them under a namespace `.generating`.

In [11]:
mfrm_1 = rp.MFRM(mfrm_sim_1)

You may wish to create a simulation based on specified, known item locations, thresholds, person locations, and/or true `lambda_r`/`omega_r` values. This may be done by passing lists to the `manual_items`, `manual_thresholds`, `manual_persons`, `manual_lambda` and/or `manual_omega` arguments (in which case there is no need to pass the relevant `item_range`, `category_base`, `max_disorder`, `person_sd`, `offset`, `global_range` or `stretch_range` arguments). You may also customise the names of the items and/or persons via `manual_item_names`/`manual_person_names`.

This is what is done in the example `mfrm_sim_2` below: a set of specified, fixed item locations (6 items between -2.5 and +2.5 logits, maximum score of 5) and a set of Rasch-Andrich thresholds (summing to zero) are passed together with 5 raters' true `lambda_r`/`omega_r` values specified directly, and a random uniform distribution of person locations (between -2 and +2 logits). Raters 1-2 are neutral (`lambda_r=0`, `omega_r=1`); Rater 3 is a lenient 'central' rater (negative `lambda_r`, `omega_r>1`); Rater 4 is a severe 'extreme' rater (positive `lambda_r`, `omega_r<1`); Rater 5 is neutral in severity but mildly 'central'. For this simulation, we also set a proportion of 10% missing data by passing `missing=0.1`.

In [12]:
mfrm_sim_2 = rp.MFRM_Sim_Centrality(no_of_items=6,
                                    no_of_persons=500,
                                    no_of_facet_elements=5,
                                    max_score=5,
                                    missing=0.1,
                                    manual_persons=np.random.uniform(-2, 2, 500),
                                    manual_items=[-2.5, -1.5, -0.5, 0.5, 1.5, 2.5],
                                    manual_thresholds=[-2, -1, 0, 1, 2],
                                    manual_lambda=[0, 0, -1, 1, 0],
                                    manual_omega=[1, 1, 1.5, 0.6, 1.3],
                                    seed=42)

Save the generated response dataframe, which is stored as an attribute `mfrm_sim_2.responses`, to file, and view the first 5 lines.

In [13]:
mfrm_sim_2.responses

Item_1  Item_2  Item_3  Item_4  Item_5  Item_6
Rater_1 Person_1       4.0     4.0     2.0     2.0     3.0     0.0
        Person_2       4.0     3.0     4.0     2.0     1.0     0.0
        Person_3       3.0     1.0     2.0     1.0     0.0     1.0
        Person_4       NaN     4.0     2.0     2.0     0.0     0.0
        Person_5       4.0     4.0     2.0     3.0     1.0     0.0
...                    ...     ...     ...     ...     ...     ...
Rater_5 Person_496     4.0     4.0     4.0     2.0     3.0     1.0
        Person_497     5.0     4.0     2.0     2.0     3.0     0.0
        Person_498     3.0     3.0     2.0     2.0     NaN     1.0
        Person_499     5.0     4.0     NaN     4.0     3.0     2.0
        Person_500     5.0     NaN     4.0     4.0     3.0     1.0

[2500 rows x 6 columns]

Save the generating item, threshold and person parameters to file, and view the item locations and Rasch-Andrich thresholds.

In [14]:
mfrm_sim_2.items.to_csv('mfrm_sim_2_items.csv', header=None)
mfrm_sim_2.items

Item_1   -2.5
Item_2   -1.5
Item_3   -0.5
Item_4    0.5
Item_5    1.5
Item_6    2.5
dtype: float64

In [15]:
mfrm_sim_2.thresholds.to_csv('mfrm_sim_2_thresholds.csv', header=None)
mfrm_sim_2.thresholds

1   -2
2   -1
3    0
4    1
5    2
dtype: int64

In [16]:
mfrm_sim_2.facet_effects.to_csv('mfrm_sim_2_facet_effects.csv')
mfrm_sim_2.facet_effects.head()

,1,2,3,4,5
Rater_1,0.0,0.0,0.0,0.0,0.0
Rater_2,0.0,0.0,0.0,0.0,0.0
Rater_3,-2.0,-1.5,-1.0,-0.5,0.0
Rater_4,1.8,1.4,1.0,0.6,0.2
Rater_5,-0.6,-0.3,0.0,0.3,0.6


In [17]:
mfrm_sim_2.persons.to_csv('mfrm_sim_2_persons.csv', header=None)
mfrm_sim_2.persons.head()

Person_1    0.271753
Person_2   -0.248627
Person_3   -1.653141
Person_4   -0.126363
Person_5   -0.620761
dtype: float64

View the true `lambda_r`/`omega_r` values specified above:

In [18]:
pd.DataFrame({'lambda': mfrm_sim_2.lambda_, 'omega': mfrm_sim_2.omega})

,lambda,omega
Rater_1,0.0,1.0
Rater_2,0.0,1.0
Rater_3,-1.0,1.5
Rater_4,1.0,0.6
Rater_5,0.0,1.3


View `max_score`.

In [19]:
mfrm_sim_2.max_score

5

Create an object, `mfrm_2`, of the class `MFRM` from the response dataframe for analysis.

In [20]:
mfrm_2 = rp.MFRM(mfrm_sim_2)

The two `MFRM` objects `mfrm_1` and `mfrm_2` are now available for analysis and, where appropriate, comparison of the recovered estimates with the generating estimates (including `mfrm_1.calibrate_centrality()`'s `lambda_centrality`/`omega_centrality` against `mfrm_sim_1.lambda_`/`mfrm_sim_1.omega`). See the example `Centrality MFRM` notebook for details on how to run an `MFRM` analysis.